# Dataset Processing

FGS     (n_frames, 135000, 32, 32)   ->  (n_frames, 35, 16, 1)

AIRS    (n_frames, 11250, 32, 356)   ->  (n_frames, 35, 16, 282)

Then concat to one (n_frames, 35, 16, 283)

In [2]:
import numpy as np
import pandas as pd
import os
import glob
from tqdm.auto import tqdm
import itertools
from astropy.stats import sigma_clip
import json
from scipy.optimize import minimize
from astropy.stats import sigma_clip
from scipy.signal import savgol_filter
from concurrent.futures import ThreadPoolExecutor, as_completed

temporal_bins = 30
config = {
    "IS_TRAIN": False,
    "DATA_PATH": './data',
    "DATASET": "test",
    "OUTPUT_DIR": "./processed_data",
    "SCALE": 0.95,
    "SIGMA": 0.0006,
    "CUT_INF": 39,
    "CUT_SUP": 321,
    "TEMPORAL_BINS": 30,
    
    "SENSOR_CONFIG": {
        "AIRS-CH0": {
            "raw_shape": [11250, 32, 356],
            "calibrated_shape": [1, 32, 282],
            "dark_shape": (32, 356),
            "dead_shape": (32, 356),
            "flat_shape": (32, 356),
            "linear_corr_shape": (6, 32, 356),
            "dt_pattern": (0.1, 4.5), 
            "binning": temporal_bins
        },
        "FGS1": {
            "raw_shape": [135000, 32, 32],
            "calibrated_shape": [1, 32, 32],
            "dark_shape": (32, 32),
            "dead_shape": (32, 32),
            "flat_shape": (32, 32),
            "linear_corr_shape": (6, 32, 32),
            "dt_pattern": (0.1, 0.1),
            "binning": temporal_bins * 12
        }
    },
    "MODEL_PHASE_DETECTION_SLICE": slice(30, 140),
    "MODEL_OPTIMIZATION_DELTA": 7,
    "MODEL_POLYNOMIAL_DEGREE": 3,
    "N_JOBS": 4
}

In [3]:
class TransitModel:
    def __init__(self, config):
        self.cfg = config

    def _phase_detector(self, signal):
        search_slice = self.cfg['MODEL_PHASE_DETECTION_SLICE']
        min_index = np.argmin(signal[search_slice]) + search_slice.start
        signal1 = signal[:min_index]
        signal2 = signal[min_index:]
        grad1 = np.gradient(signal1)
        grad1 /= grad1.max()
        grad2 = np.gradient(signal2)
        grad2 /= grad2.max()
        phase1 = np.argmin(grad1)
        phase2 = np.argmax(grad2) + min_index
        return phase1, phase2
    
    def _objective_function(self, s, signal, phase1, phase2):
        delta = self.cfg['MODEL_OPTIMIZATION_DELTA']
        power = self.cfg['MODEL_POLYNOMIAL_DEGREE']
        if phase1 - delta <= 0 or phase2 + delta >= len(signal) or phase2 - delta - (phase1 + delta) < 5:
            delta = 2
        y = np.concatenate([
            signal[: phase1 - delta],
            signal[phase1 + delta : phase2 - delta] * (1 + s),
            signal[phase2 + delta :]
        ])
        x = np.arange(len(y))
        coeffs = np.polyfit(x, y, deg=power)
        poly = np.poly1d(coeffs)
        error = np.abs(poly(x) - y).mean()
        return error

    def predict(self, single_preprocessed_signal):
        signal_1d = single_preprocessed_signal[:, 1:].mean(axis=1)
        signal_1d = savgol_filter(signal_1d, 20, 2)
        
        phase1, phase2 = self._phase_detector(signal_1d)

        phase1 = max(self.cfg['MODEL_OPTIMIZATION_DELTA'], phase1)
        phase2 = min(len(signal_1d) - self.cfg['MODEL_OPTIMIZATION_DELTA'] - 1, phase2)

        result = minimize(
            fun=self._objective_function,
            x0=[0.0001],
            args=(signal_1d, phase1, phase2),
            method="Nelder-Mead"
        )
        
        return result.x[0]

    def predict_all(self, preprocessed_signals):
        predictions = [
            self.predict(preprocessed_signal)
            for preprocessed_signal in tqdm(preprocessed_signals)
]
        return np.array(predictions) * self.cfg['SCALE']

In [10]:
class ArielDataProcessor:
    def __init__(self, data_path="./data", output_path="./processed_data", is_train=True, n_jobs=1):
        self.data_path = data_path
        self.output_path = output_path
        self.is_train = is_train
        self.n_jobs = n_jobs
        
        os.makedirs(output_path, exist_ok=True)
        self.adc_info = pd.read_csv(f"{data_path}/adc_info.csv")
        star_info = pd.read_csv(f"{data_path}/{'train' if self.is_train else 'test'}_star_info.csv", index_col='planet_id')
        self.planet_ids = star_info.index.astype(int).tolist()
        
    def robust_downsample(self, array, factor, mode='mean'):
        n_frames = array.shape[0] // factor
        if n_frames == 0:
            return None
        result = []
        for i in range(n_frames):
            chunk = array[i*factor:(i+1)*factor]
            if hasattr(chunk, 'mask'):
                if mode == 'median':
                    val = np.ma.median(chunk, axis=0)
                else:
                    val = np.ma.mean(chunk, axis=0)
                if hasattr(val, 'filled'):
                    if val.mask.any():
                        unmasked_data = val.data[~val.mask]
                        if len(unmasked_data) > 0:
                            fill_value = np.median(unmasked_data)
                        else:
                            fill_value = 0.0
                    else:
                        fill_value = 0.0
                    val = val.filled(fill_value)
            else:
                if mode == 'median':
                    val = np.nanmedian(chunk, axis=0)
                else:
                    val = np.nanmean(chunk, axis=0)
                val = np.nan_to_num(val, nan=0.0, posinf=0.0, neginf=0.0)
            result.append(val)
        final_result = np.array(result)
        final_result = np.nan_to_num(final_result, nan=0.0, posinf=0.0, neginf=0.0)
        return final_result
        
    def adc_convert(self, signal, gain=0.4369, offset=-1000):
        signal = signal.astype(np.float64)
        signal /= gain
        signal += offset
        return signal
        
    def mask_hot_dead(self, signal, dead, dark):
        hot = sigma_clip(dark, sigma=5, maxiters=5).mask
        hot = np.tile(hot, (signal.shape[0], 1, 1))
        dead = np.tile(dead, (signal.shape[0], 1, 1))
        signal = np.ma.masked_where(dead, signal)
        signal = np.ma.masked_where(hot, signal)
        return signal
        
    def apply_linear_corr(self, linear_corr, clean_signal):
        linear_corr = np.flip(linear_corr, axis=0)
        linear_corr = np.nan_to_num(linear_corr, nan=0.0, posinf=1.0, neginf=0.0)
        for x, y in itertools.product(range(clean_signal.shape[1]), range(clean_signal.shape[2])):
            try:
                coeffs = linear_corr[:, x, y]
                if np.allclose(coeffs, 0):
                    continue
                poli = np.poly1d(coeffs)
                old_vals = clean_signal[:, x, y].copy()
                new_vals = poli(old_vals)
                new_vals = np.nan_to_num(new_vals, nan=old_vals, posinf=old_vals, neginf=old_vals)
                clean_signal[:, x, y] = new_vals
            except Exception:
                continue
        return clean_signal
        
    def clean_dark(self, signal, dead, dark, dt):
        dark = np.ma.masked_where(dead, dark)
        dark = np.tile(dark, (signal.shape[0], 1, 1))
        signal = signal - dark * dt[:, np.newaxis, np.newaxis]
        return signal
        
    def get_cds(self, signal):
        if signal.shape[0] < 2:
            return np.empty((0,) + signal.shape[1:])
        cds = signal[1::2, :, :] - signal[0::2, :, :]
        return cds
    
    def correct_flat_field(self, flat, dead, signal):
        flat = flat.T
        dead = dead.T
        flat = np.ma.masked_where(dead, flat)
        flat = np.tile(flat, (signal.shape[0], 1, 1))
        signal = signal / flat
        return signal
        
    def process_single_observation(self, planet_id, obs_id, airs_downsample, fgs_downsample, mode='mean'):
        airs_signal_path = f"{self.data_path}/{'train' if self.is_train else 'test'}/{planet_id}/AIRS-CH0_signal_{obs_id}.parquet"
        fgs_signal_path = f"{self.data_path}/{'train' if self.is_train else 'test'}/{planet_id}/FGS1_signal_{obs_id}.parquet"
        if not (os.path.exists(airs_signal_path) and os.path.exists(fgs_signal_path)):
            return None
        try:
            airs_data, mean_airs_data = self.process_airs(planet_id, obs_id, airs_downsample, mode)
            if airs_data is None:
                return None
            fgs_data, mean_fgs_data = self.process_fgs(planet_id, obs_id, fgs_downsample, mode)
            if fgs_data is None:
                return None
            if airs_data.shape[0] != fgs_data.shape[0]:
                min_frames = min(airs_data.shape[0], fgs_data.shape[0])
                airs_data = airs_data[:min_frames]
                mean_airs_data = mean_airs_data[:min_frames]
                fgs_data = fgs_data[:min_frames]
                mean_fgs_data = mean_fgs_data[:min_frames]
            combined_data = np.nan_to_num(np.concatenate([fgs_data, airs_data], axis=2), nan=0.0)
            mean_combined_data = np.nan_to_num(np.concatenate([mean_fgs_data, mean_airs_data], axis=1), nan=0.0)
            output_filename = f"planet_{planet_id}_signal_{obs_id}_downsample_{airs_downsample}.npy"
            output_path = os.path.join(self.output_path, output_filename)
            np.save(output_path, combined_data)
            return mean_combined_data
        except Exception as e:
            print(f"❌ Error processing planet {planet_id}, obs {obs_id}: {e}")
            return None
        
    def process_airs(self, planet_id, obs_id, downsample_factor, mode='mean'):
        try:
            signal_df = pd.read_parquet(f"{self.data_path}/{'train' if self.is_train else 'test'}/{planet_id}/AIRS-CH0_signal_{obs_id}.parquet")
            signal = signal_df.values.astype(np.float64).reshape(signal_df.shape[0], 32, 356)
            signal = self.adc_convert(signal)
            cal_path = f"{self.data_path}/{'train' if self.is_train else 'test'}/{planet_id}/AIRS-CH0_calibration_{obs_id}"
            flat = pd.read_parquet(f"{cal_path}/flat.parquet").values.astype(np.float64).reshape(32, 356)
            dark = pd.read_parquet(f"{cal_path}/dark.parquet").values.astype(np.float64).reshape(32, 356)
            dead = pd.read_parquet(f"{cal_path}/dead.parquet").values.astype(np.float64).reshape(32, 356)
            linear_corr = pd.read_parquet(f"{cal_path}/linear_corr.parquet").values.astype(np.float64).reshape(6, 32, 356)
            signal = signal[:, :, 39:321]
            flat   = flat[:,   39:321]
            dark   = dark[:,   39:321]
            dead   = dead[:,   39:321]
            linear_corr = linear_corr[:, :, 39:321]
            np.maximum(signal, 0, out=signal)
            signal_roi = signal[:, 8:24, :]
            linear_corr_roi = linear_corr[:, 8:24, :]
            corrected_roi = self.apply_linear_corr(linear_corr_roi, signal_roi.copy())
            signal[:, 8:24, :] = corrected_roi
            dt_airs = np.ones(len(signal)) * 0.1
            dt_airs[1::2] = 0.2
            signal = self.clean_dark(signal, dead, dark, dt_airs)
            cds_signal = self.get_cds(signal)
            if cds_signal.shape[0] == 0:
                return None
            
            mean_cds_signal = np.nanmean(cds_signal, axis=1)
            cds_cropped = cds_signal[:, 8:24, :]
            cds_cropped = cds_cropped[:, :, ::-1]

            mean_downsampled = np.array([np.nanmean(mean_cds_signal[j*30:(j+1)*30], axis=0)
                               for j in range(cds_signal.shape[0] // 30)])
            mean_downsampled = np.clip(mean_downsampled,
                    np.nanpercentile(mean_downsampled, 5.0, axis=1, keepdims=True), 
                    np.nanpercentile(mean_downsampled, 95.0, axis=1, keepdims=True))
            var = np.nanvar(mean_downsampled, axis=0, ddof=1)
            med = np.nanmedian(var)
            safe_var = np.where(~np.isfinite(var) | (var <= 0), med if (np.isfinite(med) and med > 0) else 1.0, var)
            w = 1.0 / safe_var
            lo, hi = np.nanpercentile(w, 5.0), np.nanpercentile(w, 95.0)
            if np.isfinite(lo) and np.isfinite(hi) and lo < hi:
                w = np.clip(w, lo, hi)
            M = mean_downsampled.shape[1]
            s = np.nansum(w)
            w = w * (M / s) if (np.isfinite(s) and s > 0) else np.ones_like(w)
            mean_downsampled *= w[None, :]
            
            downsampled = self.robust_downsample(cds_cropped, downsample_factor, mode)
            if downsampled is None:
                return None, None
            return downsampled, mean_downsampled
        except Exception as e:
            print(f"❌ Error processing AIRS for planet {planet_id}, obs {obs_id}: {e}")
            return None, None

    def process_fgs(self, planet_id, obs_id, downsample_factor, mode='mean'):
        try:
            signal_df = pd.read_parquet(f"{self.data_path}/{'train' if self.is_train else 'test'}/{planet_id}/FGS1_signal_{obs_id}.parquet")
            signal = signal_df.values.astype(np.float64).reshape(signal_df.shape[0], 32, 32)
            signal = self.adc_convert(signal)
            cal_path = f"{self.data_path}/{'train' if self.is_train else 'test'}/{planet_id}/FGS1_calibration_{obs_id}"
            flat = pd.read_parquet(f"{cal_path}/flat.parquet").values.astype(np.float64).reshape(32, 32)
            dark = pd.read_parquet(f"{cal_path}/dark.parquet").values.astype(np.float64).reshape(32, 32)
            dead = pd.read_parquet(f"{cal_path}/dead.parquet").values.astype(np.float64).reshape(32, 32)
            linear_corr = pd.read_parquet(f"{cal_path}/linear_corr.parquet").values.astype(np.float64).reshape(6, 32, 32)
            np.maximum(signal, 0, out=signal)
            signal = self.apply_linear_corr(linear_corr, signal)
            dt_fgs = np.ones(len(signal)) * 0.1
            dt_fgs[1::2] = 0.2
            signal = self.clean_dark(signal, dead, dark, dt_fgs)
            cds_signal = self.get_cds(signal)
            if cds_signal.shape[0] == 0:
                return None
            if hasattr(cds_signal, 'mask'):
                cds_averaged = np.ma.mean(cds_signal, axis=2)
            else:
                cds_averaged = np.nanmean(cds_signal, axis=2)
                cds_averaged = np.nan_to_num(cds_averaged, nan=0.0)
            cds_cropped = cds_averaged[:, 8:24]
            cds_cropped = cds_cropped[:, :, np.newaxis]
            mean_cds_signal = np.nanmean(cds_averaged, axis=1)[:, np.newaxis]
            
            mean_downsampled = np.array([np.nanmean(mean_cds_signal[j*360:(j+1)*360], axis=0)
                               for j in range(cds_signal.shape[0] // 360)])
            mean_downsampled = np.clip(mean_downsampled,
                    np.nanpercentile(mean_downsampled, 5.0, axis=1, keepdims=True), 
                    np.nanpercentile(mean_downsampled, 95.0, axis=1, keepdims=True))
            downsampled = self.robust_downsample(cds_cropped, downsample_factor, mode)
            if downsampled is None:
                return None, None
            return downsampled, mean_downsampled
        except Exception as e:
            print(f"❌ Error processing FGS for planet {planet_id}, obs {obs_id}: {e}")
            return None, None

    def process_all_data(self, airs_factors=40, fgs_ratio=12, mode='mean'):
        """
        Process all planets/obs with parallel processing.

        Parameters:
        - airs_factors: int or list[int]
            Temporal downsampling factor(s) for AIRS (e.g., 40 or [40,80,100]).
        - fgs_ratio: int
            FGS factor is computed as airs_factor * fgs_ratio (default 12 to match timing).
        - mode: 'mean' or 'median'
        """
        # Normalize factors to a list
        if isinstance(airs_factors, int):
            airs_factors = [airs_factors]
        airs_factors = list(airs_factors)

        tasks = []
        for planet_id in self.planet_ids:
            for obs_id in [0, 1]:
                airs_path = f"{self.data_path}/{'train' if self.is_train else 'test'}/{planet_id}/AIRS-CH0_signal_{obs_id}.parquet"
                fgs_path = f"{self.data_path}/{'train' if self.is_train else 'test'}/{planet_id}/FGS1_signal_{obs_id}.parquet"
                if not (os.path.exists(airs_path) and os.path.exists(fgs_path)):
                    continue
                for af in airs_factors:
                    ff = int(af * fgs_ratio)
                    tasks.append({
                        'planet_id': planet_id,
                        'obs_id': obs_id,
                        'airs_downsample': af,
                        'fgs_downsample': ff,
                        'mode': mode
                    })

        print(f"Found {len(tasks)} observation-factor tasks to process...")

        def process_task(task):
            return self.process_single_observation(
                task['planet_id'],
                task['obs_id'],
                task['airs_downsample'],
                task['fgs_downsample'],
                task['mode']
            )
        
        results = []
        if self.n_jobs is not None and self.n_jobs > 1:
            results = [None] * len(tasks)
            with ThreadPoolExecutor(max_workers=self.n_jobs) as ex:
                fut2i = {ex.submit(process_task, t): i for i, t in enumerate(tasks)}
                for fut in tqdm(as_completed(fut2i), total=len(tasks), desc="Processing observations"):
                    i = fut2i[fut]
                    try: results[i] = fut.result()
                    except Exception: results[i] = None
        else:
            for i, t in enumerate(tqdm(tasks, desc="Processing observations")):
                try: results.append(process_task(t))
                except Exception: results.append(None)

        successful_results = [r for r in results if r is not None]
        print(f"Successfully processed {len(successful_results)} out of {len(tasks)} tasks")

        transit_model = TransitModel(config)
        transit_depth = transit_model.predict_all(results)
        transit_depth = pd.DataFrame(transit_depth, index=pd.Index([task['planet_id'] for task in tasks], name="planet_id"), columns=['transit_depth'])
        transit_depth = transit_depth.groupby(level=0).mean()
        transit_depth.to_csv(f"{self.output_path}/transit_depth.csv")


        return transit_depth


In [11]:
processor = ArielDataProcessor(
    data_path=config["DATA_PATH"],
    output_path=config["OUTPUT_DIR"],
    is_train=config["IS_TRAIN"],
    n_jobs=4  # Adjust based on your CPU cores
)
print("Processing with mean downsampling...")
transit_depth = processor.process_all_data(airs_factors=(160), fgs_ratio=12, mode='mean')

# print("\nProcessing with median downsampling...")
# processor.process_all_data(airs_factors=(80, 160), fgs_ratio=12, mode='median')

print(f"\nAll done! Check {processor.output_path} for results.")

Processing with mean downsampling...
Found 20 observation-factor tasks to process...


Processing observations:   0%|          | 0/20 [00:00<?, ?it/s]

Successfully processed 20 out of 20 tasks


  0%|          | 0/20 [00:00<?, ?it/s]


All done! Check ./processed_data for results.


# Dataset

In [12]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from pathlib import Path
import pytorch_lightning as pl
from typing import Optional, Callable

class ArielPreprocessedDataset(Dataset):
    def __init__(self, 
                 preprocessed_dir: str, 
                 train_csv: str, 
                 star_info_csv: Optional[str] = None,
                 wavelengths_csv: Optional[str] = None,
                 transit_depth_csv: Optional[str] = None,
                 sigma_csv: Optional[str] = None,
                 downsample_mode: str = "40",  # "40" or "80" or "160"
                 transform: Optional[Callable] = None,
                 verbose: bool = True,
                 pad_to_T: Optional[int] = None,
                 pad_value: float = 0.0):
        self.preprocessed_dir = Path(preprocessed_dir)
        self.transform = transform
        self.verbose = verbose
        self.downsample_mode = downsample_mode
        self.pad_to_T = pad_to_T
        self.pad_value = float(pad_value)   

        # Train targets: Mu
        self.train_df = pd.read_csv(train_csv, index_col='planet_id')
        
        # Optionals
        self.star_info = pd.read_csv(star_info_csv, index_col='planet_id') if star_info_csv else None
        self.transit_depth = pd.read_csv(transit_depth_csv, index_col='planet_id') if transit_depth_csv else None
        self.sigma = pd.read_csv(sigma_csv, index_col='planet_id') if sigma_csv else None

        # Wavelengths
        self.wavelengths = None
        if wavelengths_csv and Path(wavelengths_csv).exists():
            wl = pd.read_csv(wavelengths_csv).values.flatten().astype(np.float32)
            self.wavelengths = torch.from_numpy(wl)
        else:
            self.wavelengths = torch.linspace(0.5, 7.8, 283, dtype=torch.float32)
        
        # Files passendes downsample
        pattern = f'planet_*_signal_*_downsample_{downsample_mode}.npy'
        self.processed_files = sorted(self.preprocessed_dir.glob(pattern))

        # Filtrate + meta
        self.file_map = []
        for f in self.processed_files:
            parts = f.stem.split('_')
            if "downsample" in parts[-2]:
                planet_id = int(parts[1])
                signal_idx = int(parts[3])
                self.file_map.append((planet_id, signal_idx, f))
        # Filter real train-planets only
        self.file_map = [
            (pid, sig_idx, path) for pid, sig_idx, path in self.file_map
            if pid in self.train_df.index
        ]
        if verbose:
            print(f"Loaded {len(self.file_map)} obs (downsample {downsample_mode})")

    def __len__(self):
        return len(self.file_map)

    def __getitem__(self, idx):
        planet_id, signal_idx, file_path = self.file_map[idx]
        signal_data = np.load(file_path)  # (n_times, 16, 283)
        
        if self.pad_to_T is not None:
            T, H, W = signal_data.shape
            if T > self.pad_to_T:
                signal_data = signal_data[:self.pad_to_T]
            elif T < self.pad_to_T:
                pad = np.full((self.pad_to_T - T, H, W), self.pad_value, dtype=signal_data.dtype)
                signal_data = np.concatenate([signal_data, pad], axis=0)
        # targets
        target_spectrum = self.train_df.loc[planet_id].values.astype(np.float32)  # (283,)
        # sigmas
        if self.sigma is not None and planet_id in self.sigma.index:
            fgs_sigma = self.sigma.loc[planet_id, "fgs_sigma"]
            airs_sigma = self.sigma.loc[planet_id, "airs_sigma"]
            sigma_arr = np.concatenate([[fgs_sigma], [airs_sigma]*282]).astype(np.float32)  # (283,)
        else:
            sigma_arr = np.ones(283, dtype=np.float32)

        # transit depth
        if self.transit_depth is not None and planet_id in self.transit_depth.index:
            transit_depth = self.transit_depth.loc[planet_id].values.astype(np.float32)
        else:
            transit_depth = np.zeros(1, dtype=np.float32)

        # transform
        if self.transform is not None:
            signal_data = self.transform(signal_data)
        if signal_data.dtype != np.float32:
            signal_data = signal_data.astype(np.float32, copy=False)
        signal_tensor = torch.from_numpy(signal_data)
        target_tensor = torch.from_numpy(target_spectrum)
        sigma_tensor = torch.from_numpy(sigma_arr)
        transit_depth_tensor = torch.from_numpy(transit_depth)
        wavelengths_tensor = self.wavelengths

        sample = {
            'planet_id': planet_id,
            'signal_idx': signal_idx,
            'signal_data': signal_tensor,     # (T, 16, 283)
            'target_spectrum': target_tensor, # (283,)
            'sigmas': sigma_tensor,           # (283,)
            'transit_depth': transit_depth_tensor, # (1,)
            'wavelengths': wavelengths_tensor # (283,)
        }
        if self.star_info is not None and planet_id in self.star_info.index:
            sample["star_params"] = torch.tensor(self.star_info.loc[planet_id].values.astype(np.float32))
        else:
            sample["star_params"] = torch.zeros(8, dtype=torch.float32)
        return sample


# Data Module

In [13]:
import pytorch_lightning as pl
from torch.utils.data import DataLoader, Subset
import numpy as np

def _collate_keep_numpy_to_from_numpy(batch):
    keys = batch[0].keys()
    out = {}
    for k in keys:
        vals = [b[k] for b in batch]
        if k in ('planet_id', 'signal_idx'):
            out[k] = torch.tensor(vals, dtype=torch.int64)
            continue
        if k == 'signal_data':
            same_T = all(v.shape[0] == vals[0].shape[0] for v in vals)
            out[k] = torch.stack(vals, dim=0) if same_T else vals  # (B, T, 16, 283)
            continue
        out[k] = torch.stack(vals, dim=0) if isinstance(vals[0], torch.Tensor) and vals[0].ndim == 1 else vals
    return out

class ArielDataModule(pl.LightningDataModule):
    def __init__(
        self,
        preprocessed_dir: str,
        train_csv: str,
        star_info_csv: str = None,
        wavelengths_csv: str = None,
        transit_depth_csv: str = None,
        sigma_csv: str = None,
        batch_size: int = 16,
        num_workers: int = 0,
        val_split: float = 0.2,
        transform=None,
        downsample_mode: str = "160",
        seed: int = 42,
        verbose: bool = True,
        pad_to_T: int | None = None,
        pad_value: float = 0.0,
    ):
        super().__init__()
        self.save_hyperparameters(ignore=["transform"])
        self.transform = transform

    def setup(self, stage=None):
        from pathlib import Path

        self.full_dataset = ArielPreprocessedDataset(
            preprocessed_dir=self.hparams.preprocessed_dir,
            train_csv=self.hparams.train_csv,
            star_info_csv=self.hparams.star_info_csv,
            wavelengths_csv=self.hparams.wavelengths_csv,
            transit_depth_csv=self.hparams.transit_depth_csv,
            sigma_csv=self.hparams.sigma_csv,
            downsample_mode=self.hparams.downsample_mode,
            transform=self.transform,
            verbose=self.hparams.verbose,
            pad_to_T=self.hparams.pad_to_T,
            pad_value=self.hparams.pad_value,
        )
        # Splitting (planet-wise, no leaky validation)
        planet_ids = [pid for pid, _, _ in self.full_dataset.file_map]
        unique_planets = np.array(list(set(planet_ids)))
        rng = np.random.default_rng(self.hparams.seed)
        rng.shuffle(unique_planets)
        val_size = int(len(unique_planets) * self.hparams.val_split)
        val_planets = set(unique_planets[:val_size])
        train_planets = set(unique_planets[val_size:])

        train_indices = [
            i for i, (pid, _, _) in enumerate(self.full_dataset.file_map)
            if pid in train_planets
        ]
        val_indices = [
            i for i, (pid, _, _) in enumerate(self.full_dataset.file_map)
            if pid in val_planets
        ]
        self.train_dataset = Subset(self.full_dataset, train_indices)
        self.val_dataset = Subset(self.full_dataset, val_indices)
        if self.hparams.verbose:
            print(f"Planets: {len(unique_planets)}, Train: {len(train_indices)}, Val: {len(val_indices)}")

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.hparams.batch_size,
            shuffle=True,
            num_workers=self.hparams.num_workers,
            pin_memory=True,
            persistent_workers=self.hparams.num_workers > 0,
            # drop_last=True,
            # collate_fn=_collate_keep_numpy_to_from_numpy,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.hparams.batch_size,
            shuffle=False,
            num_workers=self.hparams.num_workers,
            pin_memory=True,
            persistent_workers=self.hparams.num_workers > 0,
            # drop_last=False,
            # collate_fn=_collate_keep_numpy_to_from_numpy,
        )

In [14]:
def test_datamodule():
    """Test the datamodule with sample data"""
    
    datamodule = ArielDataModule(
        preprocessed_dir=config['OUTPUT_DIR'],
        train_csv=config['DATA_PATH'] + '/train.csv',
        star_info_csv=config['DATA_PATH'] + f'/{"train" if config["IS_TRAIN"] else "test"}_star_info.csv',
        wavelengths_csv=config['DATA_PATH'] + '/wavelengths.csv',
        transit_depth_csv=config['OUTPUT_DIR'] + '/transit_depth.csv',
        sigma_csv=config['DATA_PATH'] + '/sigma_estimates.csv',
        batch_size=4,
        num_workers=0,  # Debug mode, set >0 for speed
        val_split=0.2,
        downsample_mode="160",
        verbose=True
    )
    
    datamodule.setup()
    
    train_loader = datamodule.train_dataloader()
    print(f"Train batches: {len(train_loader)}")
    
    for batch_idx, batch in enumerate(train_loader):
        print(f"\nBatch {batch_idx}:")
        print(f"  Planet IDs: {batch['planet_id']}")
        print(f"  Signal indices: {batch['signal_idx']}")
        print(f"  Signal data shape: {batch['signal_data'].shape}")  # [B, T, 16, 283]
        print(f"  Target spectrum shape: {batch['target_spectrum'].shape}")  # [B, 283]
        print(f"  Sigmas shape: {batch['sigmas'].shape}")  # [B, 283]
        print(f"  Transit depth shape: {batch['transit_depth'].shape}")  # [B, 1]
        print(f"  Star params shape: {batch['star_params'].shape}")  # [B, 8]
        print(f"  Wavelengths shape: {batch['wavelengths'].shape}")  # [283]
        if batch_idx == 0:
            break
    
    val_loader = datamodule.val_dataloader()
    print(f"Val batches: {len(val_loader)}")
    
    return datamodule

# To use:
datamodule = test_datamodule()


Loaded 20 obs (downsample 160)
Planets: 17, Train: 17, Val: 3
Train batches: 5

Batch 0:
  Planet IDs: tensor([1056934230, 1024292144, 1049092982, 1048114509])
  Signal indices: tensor([0, 0, 0, 0])
  Signal data shape: torch.Size([4, 35, 16, 283])
  Target spectrum shape: torch.Size([4, 283])
  Sigmas shape: torch.Size([4, 283])
  Transit depth shape: torch.Size([4, 1])
  Star params shape: torch.Size([4, 8])
  Wavelengths shape: torch.Size([4, 283])
Val batches: 1


# Conformer Architecture for Mu Prediction

In [15]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl

# ---------- Metrics ----------
def gaussian_logpdf(y, mu, sigma, eps=1e-12):
    y = y.float(); mu = mu.float(); sigma = torch.clamp(sigma.float(), min=eps)
    return -0.5*math.log(2*math.pi) - torch.log(sigma) - 0.5*((y - mu)/sigma)**2

def ariel_gll_metric(
    y_true, y_pred, sigma_user,
    naive_mean=0.01468902, naive_sigma=0.01066135,
    fgs_sigma_true=1e-6, airs_sigma_true=1e-5,
    fgs_weight=57.846, eps=1e-12
):
    y_true = y_true.float(); y_pred = y_pred.float(); sigma_user = sigma_user.float()
    sigma_true = torch.full_like(y_true, airs_sigma_true)
    sigma_true[:, 0] = fgs_sigma_true

    gll_pred = gaussian_logpdf(y_true, y_pred, sigma_user, eps=eps)
    gll_true = gaussian_logpdf(y_true, y_true, sigma_true, eps=eps)
    gll_mean = gaussian_logpdf(
        y_true,
        torch.full_like(y_true, naive_mean, dtype=torch.float32),
        torch.full_like(y_true, naive_sigma, dtype=torch.float32),
        eps=eps
    )

    denom = torch.clamp(gll_true - gll_mean, min=eps)
    ind_scores = (gll_pred - gll_mean) / denom
    w = torch.ones_like(ind_scores); w[:, 0] = fgs_weight
    score = (ind_scores * w).sum() / w.sum()
    return torch.clamp(score, min=None, max=1.0)
    
class StarEncoder(nn.Module):
    def __init__(self, d_model=256, in_dim=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, d_model // 2), 
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(d_model // 2, d_model),
            nn.LayerNorm(d_model)
        )
    def forward(self, x):
        return self.net(x)

class TransitEncoder(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, d_model), 
            nn.GELU(), 
            nn.LayerNorm(d_model)
        )
    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(1)
        return self.net(x)

# ---------- Tokenizers & Positional Encodings (unchanged core, kept stable) ----------
class TimePositionalEncoding(nn.Module):
    def __init__(self, d_model, max_t=4096): 
        super().__init__()
        self.pe_t = nn.Parameter(torch.randn(max_t, d_model)*0.02)
    def forward(self, x): 
        B,T,N,d = x.shape
        return x + self.pe_t[:T].view(1,T,1,d)

class TokenPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_n=2048): 
        super().__init__()
        self.pe_n = nn.Parameter(torch.randn(max_n, d_model)*0.02)  # CHANGED: max_n up for larger token grids
    def forward(self, x): 
        B,T,N,d = x.shape
        return x + self.pe_n[:N].view(1,1,N,d)

# ---------- Sensor-type embedding ----------
class SensorTypeEmbedding(nn.Module):
    def __init__(self, d_model): 
        super().__init__()
        self.emb = nn.Embedding(2, d_model)
    def forward(self, x, sensor_id: int): 
        return x + self.emb.weight[sensor_id].view(1,1,1,-1)


class AIRSTokenizer(nn.Module):
    def __init__(self, d_model=256, out_tokens_hw=(4,36)):
        super().__init__()
        h_out, w_out = out_tokens_hw
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, (3,7), stride=(2,4), padding=(1,3)),
            nn.GroupNorm(8, 64), nn.GELU(),
            nn.Conv2d(64, 128, (3,5), stride=(2,2), padding=(1,2)),
            nn.GroupNorm(8, 128), nn.GELU(),
            nn.Conv2d(128, d_model, 3, padding=1),
            nn.GroupNorm(16, d_model), nn.GELU()
        )
        self.h_out, self.w_out = h_out, w_out
    def forward(self, x):
        B,T,H,W = x.shape
        f = self.net(x.reshape(B*T,1,H,W))
        f = F.adaptive_avg_pool2d(f, (self.h_out,self.w_out))
        d,h,w = f.size(1), f.size(2), f.size(3)
        return f.permute(0,2,3,1).contiguous().view(B,T,h*w,d)

class FGSTokenizer(nn.Module):
    def __init__(self, d_model=256, out_tokens_len=4): 
        super().__init__()
        self.out_len = out_tokens_len
        self.net = nn.Sequential(
            nn.Conv1d(1,64,3,stride=2,padding=1), nn.GroupNorm(8,64), nn.GELU(),
            nn.Conv1d(64,128,3,stride=2,padding=1), nn.GroupNorm(8,128), nn.GELU(),
            nn.Conv1d(128,d_model,3,padding=1), nn.GroupNorm(16,d_model), nn.GELU()
        )
    def forward(self, x):
        B,T,H,W = x.shape
        f = self.net(x.view(B*T,1,H))
        f = F.adaptive_avg_pool1d(f, self.out_len)
        return f.permute(0,2,1).contiguous().view(B,T,self.out_len,-1)
    
# ---------- Conformer building blocks ----------
class Swish(nn.Module):
    def forward(self, x): return x * torch.sigmoid(x)

class ConformerConvModule(nn.Module):
    def __init__(self, d_model, kernel_size=31, expansion=2):
        super().__init__()
        inner = d_model * expansion
        self.pw1 = nn.Conv1d(d_model, inner, 1)
        self.glu = nn.GLU(dim=1)
        self.dw = nn.Conv1d(inner//2, inner//2, kernel_size, padding=kernel_size//2, groups=inner//2)
        self.gn = nn.GroupNorm(8, inner//2)
        self.act = Swish()
        self.pw2 = nn.Conv1d(inner//2, d_model, 1)
    def forward(self, x):  # x: (B,L,d)
        x1 = x.transpose(1,2)           # (B,d,L)
        y = self.pw1(x1)
        y = self.glu(y)                 # (B,inner/2,L)
        y = self.dw(y)
        y = self.gn(y)
        y = self.act(y)
        y = self.pw2(y).transpose(1,2)  # (B,L,d)
        return y

class ConformerFF(nn.Module):
    # Macaron FFN with dropout
    def __init__(self, d_model, expansion=4, dropout=0.1):
        super().__init__()
        inner = d_model * expansion
        self.net = nn.Sequential(
            nn.Linear(d_model, inner), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(inner, d_model),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class ConformerBlock(nn.Module):
    def __init__(self, d_model=256, nhead=8, ff_expansion=4, dropout=0.1, conv_kernel=31):
        super().__init__()
        self.ff1 = ConformerFF(d_model, ff_expansion, dropout)
        self.ln_ff1 = nn.LayerNorm(d_model)
        self.mha = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.ln_mha = nn.LayerNorm(d_model)
        self.conv = ConformerConvModule(d_model, kernel_size=conv_kernel)
        self.ln_conv = nn.LayerNorm(d_model)
        self.ff2 = ConformerFF(d_model, ff_expansion, dropout)
        self.ln_ff2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.scale = 0.5  # macaron-style scaling
    def forward(self, x, attn_mask=None):
        x = x + self.scale * self.dropout(self.ff1(self.ln_ff1(x)))
        y, _ = self.mha(self.ln_mha(x), self.ln_mha(x), self.ln_mha(x), attn_mask=attn_mask, need_weights=False)
        x = x + self.dropout(y)
        x = x + self.dropout(self.conv(self.ln_conv(x)))
        x = x + self.scale * self.dropout(self.ff2(self.ln_ff2(x)))
        return x

class ConformerEncoder(nn.Module):
    def __init__(self, d_model=256, nhead=8, layers=6, dropout=0.1):
        super().__init__()
        self.blocks = nn.ModuleList([ConformerBlock(d_model, nhead, dropout=dropout) for _ in range(layers)])
        self.ln_out = nn.LayerNorm(d_model)
    def forward(self, x, attn_mask=None):
        for blk in self.blocks: x = blk(x, attn_mask=attn_mask)
        return self.ln_out(x)

# ---------- Query head ----------
class WavelengthQueryHead(nn.Module):
    def __init__(self, d_model=256, out_dim=283):
        super().__init__()
        self.queries = nn.Parameter(torch.randn(out_dim, d_model) * 0.02)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.mu_proj = nn.Sequential(nn.Linear(d_model, d_model//2), nn.GELU(), nn.Dropout(0.1), nn.Linear(d_model//2, 1))
    def forward(self, tokens):
        B,L,d = tokens.shape
        Q = self.queries.unsqueeze(0).expand(B, -1, -1)
        K = self.k_proj(tokens); V = self.v_proj(tokens)
        scores = torch.matmul(Q, K.transpose(1,2)) / math.sqrt(d)
        scores = torch.clamp(scores, -50, 50)
        attn = torch.softmax(scores, dim=-1)
        ctx = torch.matmul(attn, V)
        return self.mu_proj(ctx).squeeze(-1)

# ---------- Full Conformer model with AIRS+FGS concat & self-attn ----------
class ArielConformer(nn.Module):
    def __init__(self, d_model=256, nhead=8, layers=6, pre_fuse_layers=2, airs_tokens_hw=(4,36), fgs_tokens_len=4):
        super().__init__()
        self.d_model = d_model
        self.airs_tok = AIRSTokenizer(d_model=d_model, out_tokens_hw=airs_tokens_hw)
        self.fgs_tok  = FGSTokenizer(d_model=d_model, out_tokens_len=fgs_tokens_len)
        self.sensor_emb = SensorTypeEmbedding(d_model)  # NEW
        self.time_pe  = TimePositionalEncoding(d_model)
        self.token_pe = TokenPositionalEncoding(d_model, max_n=airs_tokens_hw[0]*airs_tokens_hw[1]+fgs_tokens_len)
        self.star_enc = StarEncoder(d_model)
        self.td_enc   = TransitEncoder(d_model)
        self.pre_fuse = ConformerEncoder(d_model=d_model, nhead=nhead, layers=pre_fuse_layers, dropout=0.1)
        self.encoder  = ConformerEncoder(d_model=d_model, nhead=nhead, layers=layers, dropout=0.1)
        self.head     = WavelengthQueryHead(d_model=d_model, out_dim=283)

    def forward(self, signal_data, star_params, transit_depth):
        # Split sensors
        fgs = signal_data[..., :1]
        airs = signal_data[..., 1:]
        # Tokenize
        a_tok, f_tok = self.airs_tok(airs), self.fgs_tok(fgs)
        # Sensor tags (0=AIRS, 1=FGS)
        a_tok, f_tok = self.sensor_emb(a_tok, sensor_id=0), self.sensor_emb(f_tok, sensor_id=1)
        # Concat sensors along token axis (pure self-attn afterwards)
        x = torch.cat([a_tok, f_tok], dim=2)  # single stream for self-attn
        # Conditioning
        cond = self.star_enc(star_params) + self.td_enc(transit_depth)  # NOTE: if tracing, move enc to __init__
        # NOTE: for strict performance, keep star_enc/td_enc in __init__; shown in-line to stress addition
        x = x + cond.view(-1,1,1,self.d_model)
        # Positional encodings
        x = self.time_pe(x)
        x = self.token_pe(x)
        # Flatten (T,N)->L and encode with Conformer
        B,T,N,d = x.shape
        seq = x.reshape(B, T*N, d)

        seq = self.pre_fuse(seq)         
        enc = self.encoder(seq)                   
        mu  = self.head(enc)
        return mu

# ---------- Lightning with NLL + GLL logging ----------
class ArielConformerGllLightning(pl.LightningModule):
    def __init__(self, d_model=256, nhead=8, layers=6, pre_fuse_layers=2,
                 lr=1e-4, weight_decay=1e-3, lr_patience=5,
                 input_scale=0.1,
                 naive_mean=0.01468902, naive_sigma=0.01066135):
        super().__init__()
        self.save_hyperparameters()
        self.model = ArielConformer(d_model=d_model, nhead=nhead, layers=layers, pre_fuse_layers=pre_fuse_layers)
        self.input_scale = input_scale
        self.naive_mean  = naive_mean
        self.naive_sigma = naive_sigma
        self.lr_patience = lr_patience

    @staticmethod
    def nll_gaussian(y_true: torch.Tensor, y_pred: torch.Tensor, sigma_fixed: torch.Tensor) -> torch.Tensor:
        sigma = torch.clamp(sigma_fixed, min=1e-12)
        return (0.5*math.log(2*math.pi) + torch.log(sigma) + 0.5*((y_true - y_pred)/sigma)**2).mean()

    def forward(self, signal_data, star_params, transit_depth):
        return self.model(signal_data * self.input_scale, star_params, transit_depth)

    def _shared_step(self, batch, stage):
        mu_pred = self(batch['signal_data'], batch['star_params'], batch['transit_depth'])
        target, sigmas = batch['target_spectrum'], batch['sigmas']
        # nll_loss = self.nll_gaussian(target, mu_pred, sigmas)
        mse_loss = F.mse_loss(mu_pred, target, reduction='mean')
        rmse = torch.sqrt(mse_loss)
        gll_score = ariel_gll_metric(target, mu_pred, sigmas, naive_mean=self.naive_mean, naive_sigma=self.naive_sigma)
        mae = F.l1_loss(mu_pred, target, reduction='mean')
        metrics = {f'{stage}_mse': mse_loss, f'{stage}_rmse': rmse, f'{stage}_gll': gll_score, f'{stage}_mae': mae}
        return mse_loss, metrics

    def training_step(self, batch, batch_idx):
        loss, m = self._shared_step(batch, 'train')
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('train_gll',  m['train_gll'],  on_epoch=True, prog_bar=True)
        self.log('train_mse',  m['train_mse'],  on_epoch=True, prog_bar=True)
        self.log('train_rmse', m['train_rmse'], on_epoch=True)
        self.log('train_mae',  m['train_mae'],  on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, m = self._shared_step(batch, 'val')
        self.log('val_loss', loss, on_epoch=True, prog_bar=True, logger=True)
        self.log('val_gll',  m['val_gll'],  on_epoch=True, prog_bar=True, logger=True)
        self.log('val_mse',  m['val_mse'],  on_epoch=True, prog_bar=True, logger=True)
        self.log('val_rmse', m['val_rmse'], on_epoch=True, logger=True)
        self.log('val_mae',  m['val_mae'],  on_epoch=True, logger=True)
        return m['val_gll']

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=self.lr_patience, threshold=1e-9)
        return {'optimizer': opt, 'lr_scheduler': {'scheduler': sch, 'interval': 'epoch', 'monitor': 'val_rmse'}}

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger, TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor

def train_ariel_conformer(preprocessed_dir='./processed_data', data_dir='./data', batch_size=16, downsample_mode="80", max_epochs=50, learning_rate=1e-4,
        lr_patience=6, input_scale=0.1, num_workers=0, accumulate_grad_batches=1, pad_to_T=None, conformer_model=ArielConformerGllLightning, checkpoint: Optional[str] = None):
    # Create datamodule with new parameters
    datamodule = ArielDataModule(
        preprocessed_dir=preprocessed_dir,
        train_csv=data_dir + '/train.csv',
        star_info_csv=data_dir + '/train_star_info.csv',
        wavelengths_csv=data_dir + '/wavelengths.csv',
        transit_depth_csv=data_dir+'/train_transit_depth.csv',
        sigma_csv=data_dir+'/sigma_estimates.csv',
        batch_size=batch_size,
        num_workers=num_workers,
        val_split=0.2,
        downsample_mode=downsample_mode,
        verbose=True,
        pad_to_T=pad_to_T,
        seed=42
    )
    
    datamodule.setup()

    model = conformer_model(
        d_model=128,
        layers=3,
        nhead=8,
        lr=learning_rate,
        weight_decay=1e-3,
        lr_patience=lr_patience,
        input_scale=input_scale
    ) if checkpoint is None else conformer_model.load_from_checkpoint(checkpoint)
    
    # register_nan_hooks(model)
    
    # Setup callbacks
    checkpoint_callback = ModelCheckpoint(
        dirpath='./checkpoints',
        filename='ariel-sensor-{epoch:02d}-{val_rmse:.6f}-{val_gll:.4f}',
        monitor='val_rmse',
        mode='min',
        save_top_k=3,
        save_last=True
    )
    
    early_stopping = EarlyStopping(
        monitor='val_rmse',
        patience=max_epochs//3,
        mode='min',
        # min_delta=0.0005
    )
    
    lr_monitor = LearningRateMonitor(logging_interval='epoch')
    logger = TensorBoardLogger("./logs", name="ariel_conformer")
    
    # Create trainer
    torch.set_float32_matmul_precision('high')
    trainer = pl.Trainer(
        max_epochs=max_epochs,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        logger=logger,
        callbacks=[checkpoint_callback, early_stopping, lr_monitor],
        gradient_clip_val=5.0,          # Gradient clipping
        precision='16-mixed',           # Mixed precision for speed
        log_every_n_steps=10,
        check_val_every_n_epoch=1,
        enable_progress_bar=True,
        accumulate_grad_batches=accumulate_grad_batches
    )

    trainer.fit(model, datamodule)

    best_model = ArielConformerGllLightning.load_from_checkpoint(checkpoint_callback.best_model_path)

    return best_model, trainer


model, trainer = train_ariel_conformer(
    preprocessed_dir=config['OUTPUT_DIR'],
    data_dir=config['DATA_PATH'],
    batch_size=4,
    downsample_mode="160",
    max_epochs=150,
    learning_rate=5e-5,
    num_workers=0,
    accumulate_grad_batches=4,
    pad_to_T=30,
    lr_patience=5,
    input_scale=0.1,
    # checkpoint='./checkpoints/ariel-sensor-epoch=08-val_rmse=0.000500-val_gll=0.3594.ckpt'
)
print(f"Training completed! Best model saved.")


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\fazul\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:701: Checkpoint directory D:\Documents\Машинное обучение\Kaggle\Ariel Data Challenge\checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\fazul\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_lightning\utilities\model_summary\model_summary.py:231: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name  | Type           | Params | Mode 
-------------------------------------------------
0 | model | ArielConformer | 2.9 M  | train
-------------------------------------------------
2.9 M     Trainable params
0         Non-trainable params
2.9 M     Total params
11.603    Total e

Loaded 1210 obs (downsample 160)
Planets: 1100, Train: 971, Val: 239
Loaded 1210 obs (downsample 160)
Planets: 1100, Train: 971, Val: 239


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\fazul\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
c:\Users\fazul\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

C:\Users\fazul\AppData\Roaming\Python\Python310\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [16]:
import torch
import numpy as np
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

datamodule = ArielDataModule(
    preprocessed_dir=config['OUTPUT_DIR'],
    train_csv=config['DATA_PATH'] + '/train.csv',
    star_info_csv=config['DATA_PATH'] + f'/{"train" if config["IS_TRAIN"] else "test"}_star_info.csv',
    wavelengths_csv=config['DATA_PATH'] + '/wavelengths.csv',
    transit_depth_csv=config['OUTPUT_DIR'] + '/transit_depth.csv',
    sigma_csv=config['DATA_PATH'] + '/sigma_estimates.csv',
    batch_size=64,
    num_workers=0,
    val_split=0.0,
    downsample_mode="160",
    verbose=True,
    pad_to_T=30,
)
datamodule.setup()

train_dataloader = torch.utils.data.DataLoader(
    datamodule.train_dataset, batch_size=datamodule.hparams.batch_size, shuffle=False,
    num_workers=datamodule.hparams.num_workers, pin_memory=True,
    persistent_workers=False
)


ckpts = [
    './checkpoints/ariel-sensor-epoch=08-val_rmse=0.000500-val_gll=0.3594.ckpt',
    # './checkpoints/ariel-sensor-epoch=27-val_rmse=0.000542-val_gll=0.3553.ckpt',
]
os.makedirs('./predictions', exist_ok=True)

@torch.no_grad()
def predict_mu_for_loader(lightning_model, loader):
    lightning_model.eval().to(device)
    all_pids, all_mu = [], []
    for batch in tqdm(loader, desc="Predicting μ"):
        sd = batch['signal_data'].to(device)
        sp = batch['star_params'].to(device)
        td = batch['transit_depth'].to(device)
        mu = lightning_model(sd, sp, td)  # (B, 283)
        all_mu.append(mu.cpu().numpy())
        all_pids.append(batch['planet_id'].cpu().numpy())
    mu_np  = np.concatenate(all_mu, axis=0)            # (N, 283)
    pids   = np.concatenate(all_pids, axis=0)          # (N,)
    return pids, mu_np

def save_mu_csv(pids, mu_np, out_csv):
    df = pd.DataFrame(mu_np, index=pids, columns=[f'lambda_{i}' for i in range(mu_np.shape[1])])
    df.index.name = 'planet_id'
    df_agg = df.groupby(level=0).mean()
    df_agg = df_agg.sort_index()
    df_agg.to_csv(out_csv)
    print(f"Saved {out_csv} | shape={df_agg.shape}")

pred_paths = []
for ck in ckpts:
    print(f"\nLoading {ck}")
    model = ArielConformerGllLightning.load_from_checkpoint(ck)
    pids, mu_np = predict_mu_for_loader(model, train_dataloader)
    out_csv = os.path.join('./predictions', f"mu_predictions.csv")
    save_mu_csv(pids, mu_np, out_csv)
    pred_paths.append(out_csv)

if len(pred_paths) >= 2:
    dfs = [pd.read_csv(p, index_col='planet_id') for p in pred_paths]
    # Align by index
    base_index = dfs[0].index
    dfs = [df.reindex(base_index) for df in dfs]
    mu_ens = sum(dfs) / len(dfs)
    mu_ens.to_csv('./predictions/mu_predictions.csv')
    print("Saved ensemble mean at ./predictions/mu_predictions.csv")

Loaded 20 obs (downsample 160)
Planets: 17, Train: 20, Val: 0

Loading ./checkpoints/ariel-sensor-epoch=08-val_rmse=0.000500-val_gll=0.3594.ckpt


Predicting μ:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\fazul\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:5560: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)


Saved ./predictions\mu_predictions.csv | shape=(17, 283)


In [17]:
pd.read_csv('./predictions/mu_predictions.csv', index_col='planet_id').loc[[104891231, 1010375142, 1024292144, 1029552010, 1031303815, 1042982756, 1047977648]]

,lambda_0,lambda_1,lambda_2,lambda_3,lambda_4,lambda_5,lambda_6,lambda_7,lambda_8,lambda_9,...,lambda_273,lambda_274,lambda_275,lambda_276,lambda_277,lambda_278,lambda_279,lambda_280,lambda_281,lambda_282
planet_id,,,,,,,,,,,,,,,,,,,,,
104891231,0.021442,0.021443,0.021443,0.021441,0.021441,0.021443,0.021441,0.021445,0.021442,0.021442,...,0.021445,0.021442,0.021442,0.021441,0.021444,0.021443,0.021443,0.021439,0.021443,0.021442
1010375142,0.005516,0.005517,0.005518,0.005515,0.005516,0.005517,0.005516,0.005520,0.005517,0.005516,...,0.005519,0.005516,0.005517,0.005516,0.005519,0.005517,0.005517,0.005513,0.005517,0.005517
1024292144,0.008670,0.008671,0.008672,0.008670,0.008670,0.008671,0.008670,0.008674,0.008671,0.008670,...,0.008673,0.008671,0.008671,0.008670,0.008673,0.008671,0.008671,0.008667,0.008671,0.008671
1029552010,0.010697,0.010698,0.010699,0.010696,0.010696,0.010698,0.010697,0.010701,0.010697,0.010697,...,0.010700,0.010697,0.010698,0.010696,0.010699,0.010698,0.010698,0.010694,0.010698,0.010698
1031303815,0.020266,0.020267,0.020268,0.020266,0.020266,0.020267,0.020266,0.020270,0.020266,0.020266,...,0.020269,0.020267,0.020267,0.020266,0.020268,0.020267,0.020267,0.020263,0.020267,0.020267
1042982756,0.015451,0.015452,0.015453,0.015451,0.015451,0.015452,0.015451,0.015455,0.015452,0.015451,...,0.015454,0.015452,0.015452,0.015451,0.015454,0.015452,0.015453,0.015448,0.015453,0.015452
1047977648,0.008935,0.008936,0.008937,0.008935,0.008935,0.008936,0.008935,0.008939,0.008935,0.008935,...,0.008938,0.008936,0.008936,0.008935,0.008938,0.008936,0.008936,0.008932,0.008936,0.008936


# MLP Model for Sigma Prediction

In [18]:
import os
import numpy as np
import pandas as pd
from astropy.stats import sigma_clip
from tqdm.auto import tqdm

class SignalProcessor:
    def __init__(self, config):
        self.cfg = config
        self.adc_info = pd.read_csv(f"{self.cfg['DATA_PATH']}/adc_info.csv")
        self.planet_ids = pd.read_csv(
            f'{self.cfg["DATA_PATH"]}/{self.cfg["DATASET"]}_star_info.csv',
            index_col='planet_id'
        ).index.astype(int)
        os.makedirs(self.cfg['OUTPUT_DIR'], exist_ok=True)  # e.g. "./processed_data_features"

    def _apply_linear_corr(self, linear_corr, signal):
        # Horner scheme for polynomial correction
        coeffs = np.flip(linear_corr, axis=0)
        x = signal.astype(np.float64, copy=False)
        out = np.empty_like(x, dtype=np.float64)
        out[...] = coeffs[0]
        for k in range(1, coeffs.shape[0]):
            np.multiply(out, x, out=out)
            out += coeffs[k]
        return out.astype(signal.dtype, copy=False)

    def _calibrate_single_signal(self, planet_id, sensor, obs_id):
        sensor_cfg = self.cfg['SENSOR_CONFIG'][sensor]

        # Paths
        base = f"{self.cfg['DATA_PATH']}/{self.cfg['DATASET']}/{planet_id}"
        sig_pq = f"{base}/{sensor}_signal_{obs_id}.parquet"
        cal_dir = f"{base}/{sensor}_calibration_{obs_id}"
        dark_pq = f"{cal_dir}/dark.parquet"
        dead_pq = f"{cal_dir}/dead.parquet"
        flat_pq = f"{cal_dir}/flat.parquet"
        lin_pq  = f"{cal_dir}/linear_corr.parquet"

        if not (os.path.exists(sig_pq) and os.path.exists(dark_pq) and
                os.path.exists(dead_pq) and os.path.exists(flat_pq) and os.path.exists(lin_pq)):
            return None  # missing files -> skip observation

        # Loading and reshaping
        signal = pd.read_parquet(sig_pq).to_numpy().reshape(sensor_cfg["raw_shape"])
        dark   = pd.read_parquet(dark_pq).to_numpy().reshape(sensor_cfg["dark_shape"])
        dead   = pd.read_parquet(dead_pq).to_numpy().reshape(sensor_cfg["dead_shape"])
        flat   = pd.read_parquet(flat_pq).to_numpy().reshape(sensor_cfg["flat_shape"])
        linear_corr = pd.read_parquet(lin_pq).values.astype(np.float64).reshape(sensor_cfg["linear_corr_shape"])

        # ADC correction
        gain = self.adc_info[f"{sensor}_adc_gain"].iloc[0]
        offset = self.adc_info[f"{sensor}_adc_offset"].iloc[0]
        signal = signal / gain + offset

        # Hot/dead mask
        hot = sigma_clip(dark, sigma=5, maxiters=5).mask

        # AIRS: spectral crop
        if sensor == "AIRS-CH0":
            ci, cs = self.cfg['CUT_INF'], self.cfg['CUT_SUP']
            signal = signal[:, :, ci:cs]
            linear_corr = linear_corr[:, :, ci:cs]
            dark = dark[:, ci:cs]
            dead = dead[:, ci:cs]
            flat = flat[:, ci:cs]
            hot = hot[:, ci:cs]

        # FGS: ROI crop (y0:y1, x0:x1)
        if sensor == "FGS1":
            y0, y1, x0, x1 = 10, 22, 10, 22
            signal = signal[:, y0:y1, x0:x1]
            dark   = dark[y0:y1, x0:x1]
            dead   = dead[y0:y1, x0:x1]
            flat   = flat[y0:y1, x0:x1]
            linear_corr = linear_corr[:, y0:y1, x0:x1]
            hot    = hot[y0:y1, x0:x1]

        # Non negative
        np.maximum(signal, 0, out=signal)

        # Linearity correction
        if sensor == "FGS1":
            signal = self._apply_linear_corr(linear_corr, signal)
        elif sensor == "AIRS-CH0":
            sl = (slice(None), slice(10, 22), slice(None))  # T, Y, λ
            signal[sl] = self._apply_linear_corr(linear_corr[:, 10:22, :], signal[sl])
        else:
            signal = self._apply_linear_corr(linear_corr, signal)

        # Dark subtraction with dt-pattern
        base_dt, increment = sensor_cfg["dt_pattern"]
        even_scale = base_dt
        odd_scale  = base_dt + increment
        signal[::2] -= dark * even_scale
        signal[1::2] -= dark * odd_scale

        return signal

    def _preprocess_calibrated_signal(self, calibrated_signal, sensor, mode='mean'):
        sensor_cfg = self.cfg['SENSOR_CONFIG'][sensor]
        binning = sensor_cfg["binning"]

        # ROI + averaging over spatial dimensions
        if sensor == "AIRS-CH0":
            signal_roi = calibrated_signal[:, 10:22, :]              # (T, 12, λ)
        elif sensor == "FGS1":
            signal_roi = calibrated_signal[:, 10:22, 10:22]           # (T, 12, 12)
            signal_roi = signal_roi.reshape(signal_roi.shape[0], -1)  # (T, 144)
        mean_signal = np.nanmean(signal_roi, axis=1)                   # (T, λ) or (T, pixels)

        # CDS
        cds_signal = mean_signal[1::2] - mean_signal[0::2]             # (T/2, λ)

        # Temporal binning
        n_bins = cds_signal.shape[0] // binning
        if n_bins <= 0:
            return None
        if mode == 'median':
            binned = np.array([np.nanmedian(cds_signal[j*binning:(j+1)*binning], axis=0)
                               for j in range(n_bins)])
        else:
            binned = np.array([np.nanmean(cds_signal[j*binning:(j+1)*binning], axis=0)
                               for j in range(n_bins)])

        # AIRS: robust clipping per bin
        if sensor == "AIRS-CH0":
            q_lo = np.nanpercentile(binned, 5.0, axis=1, keepdims=True)
            q_hi = np.nanpercentile(binned, 95.0, axis=1, keepdims=True)
            np.clip(binned, q_lo, q_hi, out=binned)

        # FGS: to (n_bins, 1) collapse
        if sensor == "FGS1":
            binned = binned.reshape((binned.shape[0], 1))

        # AIRS: spectral weights (variance-inverse, robustly scaled)
        if sensor == "AIRS-CH0":
            var = np.nanvar(binned, axis=0, ddof=1)
            med = np.nanmedian(var)
            safe_var = np.where(~np.isfinite(var) | (var <= 0), med if (np.isfinite(med) and med > 0) else 1.0, var)
            w = 1.0 / safe_var
            lo, hi = np.nanpercentile(w, 5.0), np.nanpercentile(w, 95.0)
            if np.isfinite(lo) and np.isfinite(hi) and lo < hi:
                w = np.clip(w, lo, hi)
            M = binned.shape[1]
            s = np.nansum(w)
            w = w * (M / s) if (np.isfinite(s) and s > 0) else np.ones_like(w)
            binned *= w[None, :]

        return binned

    def _process_single_observation(self, planet_id: int, obs_id: int, mode='mean'):
        # Calibrate and preprocess sensor signals
        fgs_cal = self._calibrate_single_signal(planet_id, "FGS1", obs_id)
        airs_cal = self._calibrate_single_signal(planet_id, "AIRS-CH0", obs_id)
        if fgs_cal is None or airs_cal is None:
            return None

        fgs_pre = self._preprocess_calibrated_signal(fgs_cal, "FGS1", mode=mode)   # (n_bins, 1)
        airs_pre = self._preprocess_calibrated_signal(airs_cal, "AIRS-CH0", mode=mode) # (n_bins, 282) (after Crop)
        if fgs_pre is None or airs_pre is None:
            return None

        # Align time axes
        n = min(fgs_pre.shape[0], airs_pre.shape[0])
        fgs_pre = fgs_pre[:n]
        airs_pre = airs_pre[:n]

        # Combine: (n_bins, 1 + 282) = (n_bins, 283)
        combined = np.concatenate([fgs_pre, airs_pre], axis=1).astype(np.float32)

        # Save
        out_path = os.path.join(self.cfg['OUTPUT_DIR'], f"planet_{planet_id}_signal_{obs_id}_features.npy")
        np.save(out_path, combined)
        return out_path

    def process_all_data(self, mode='mean'):
        tasks = []
        for pid in self.planet_ids:
            for obs_id in (0, 1):
                base = f"{self.cfg['DATA_PATH']}/{self.cfg['DATASET']}/{pid}"
                if (os.path.exists(f"{base}/FGS1_signal_{obs_id}.parquet") and
                    os.path.exists(f"{base}/AIRS-CH0_signal_{obs_id}.parquet")):
                    tasks.append((int(pid), int(obs_id), mode))

        results = []
        for pid, obs_id, m in tqdm(tasks, desc="Processing observations (seq)"):
            out = self._process_single_observation(pid, obs_id, mode=m)
            if out is not None:
                results.append(out)
        return results


In [19]:
sp = SignalProcessor(config)
saved_paths = sp.process_all_data(mode='mean')
print(f"Saved {len(saved_paths)} files:")
for p in saved_paths[:5]:
    print("  ", p)

Processing observations (seq):   0%|          | 0/20 [00:00<?, ?it/s]

Saved 20 files:
   ./processed_data\planet_104891231_signal_0_features.npy
   ./processed_data\planet_1010375142_signal_0_features.npy
   ./processed_data\planet_1024292144_signal_0_features.npy
   ./processed_data\planet_1029552010_signal_0_features.npy
   ./processed_data\planet_1031303815_signal_0_features.npy


In [20]:
import os
import glob
import numpy as np
from tqdm import tqdm
from typing import List, Tuple, Dict, Optional, Iterable


class ProcessedObsLoader:

    def __init__(self, data_dir: str = "./processed_data", mmap: bool = False):
        self.data_dir = data_dir
        self.mmap = mmap

        pattern = os.path.join(data_dir, "planet_*_signal_*_features.npy")
        files = sorted(glob.glob(pattern))

        # (planet_id, obs_id, path)
        self.index: List[Tuple[int, int, str]] = []
        for p in files:
            name = os.path.basename(p)
            try:
                # Expect "planet_{pid}_signal_{obs}.npy"
                pid = int(name.split("_")[1])
                obs = int(name.split("_")[3])
            except Exception:
                continue
            self.index.append((pid, obs, p))

        # planet -> sorted obs list
        self.by_planet: Dict[int, List[int]] = {}
        for pid, obs, _ in self.index:
            self.by_planet.setdefault(pid, []).append(obs)
        for pid in self.by_planet:
            self.by_planet[pid] = sorted(set(self.by_planet[pid]))

    def __len__(self) -> int:
        return len(self.index)

    def list_planets(self) -> List[int]:
        return sorted(self.by_planet.keys())

    def list_observations(self, planet_id: int) -> List[int]:
        return self.by_planet.get(int(planet_id), [])

    def _np_load(self, path: str) -> np.ndarray:
        if self.mmap:
            return np.load(path, mmap_mode="r")
        return np.load(path)

    def load_single_observation(self, planet_id: int, obs_id: int) -> np.ndarray:
        matches = [p for (pid, obs, p) in self.index if pid == int(planet_id) and obs == int(obs_id)]
        if not matches:
            raise FileNotFoundError(f"No file for planet={planet_id}, obs={obs_id} in {self.data_dir}")
        return self._np_load(matches[0])

    def get_obs_iterator(self) -> Iterable[Tuple[int, int, np.ndarray]]:
        """Yield (planet_id, obs_id, array) lazily."""
        for pid, obs, p in self.index:
            yield pid, obs, self._np_load(p)

    def get_planet_iterator(self, planet_id: int) -> Iterable[Tuple[int, int, np.ndarray]]:
        """Yield all observations for one planet."""
        for pid, obs, p in self.index:
            if pid == int(planet_id):
                yield pid, obs, self._np_load(p)

    # Simplified: no filter / no stack; split by obs_id for train/val
    def load_all_observations(self):
        """
        Returns:
          train_transit_features: list of np.ndarray (T, 283) for obs_id == 0
          val_transit_features:   list of np.ndarray (T, 283) for obs_id == 1
          val_indices:            np.ndarray of planet_id matching val list order
        """
        train_data: List[np.ndarray] = []
        train_idx:  List[int]        = []
        val_data:   List[np.ndarray] = []
        val_idx:    List[int]        = []

        for pid, obs, p in tqdm(self.index, desc="Loading train/val by obs_id"):
            if obs == 0:
                train_data.append(self._np_load(p))
                train_idx.append(pid)
            elif obs == 1:
                val_data.append(self._np_load(p))
                val_idx.append(pid)
            else:
                # ignore other obs_ids
                continue

        return np.asarray(train_data), np.asarray(train_idx, dtype=np.int64), np.asarray(val_data), np.asarray(val_idx, dtype=np.int64)

In [21]:
loader = ProcessedObsLoader(config['OUTPUT_DIR'], mmap=False)

train_planets, train_indices, val_planets, val_indices = loader.load_all_observations()
train_planets.shape, train_indices.shape, val_planets.shape, val_indices.shape

train_labels = pd.read_csv(config['DATA_PATH'] + '/train.csv', index_col='planet_id').loc[train_indices]
val_labels = train_labels.loc[val_indices]
train_labels, val_labels = train_labels.values, val_labels.values
train_labels.shape, val_labels.shape

Loading train/val by obs_id: 100%|██████████| 20/20 [00:00<00:00, 230.26it/s]


((17, 283), (3, 283))

In [22]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.signal import savgol_filter
from scipy.optimize import minimize


class TransitModelWithFeaturesPerFreq:
    def __init__(self, config):
        self.cfg = config

    def _phase_detector(self, signal):
        search_slice = self.cfg['MODEL_PHASE_DETECTION_SLICE']
        min_index = np.argmin(signal[search_slice]) + search_slice.start

        signal1 = signal[:min_index]
        signal2 = signal[min_index:]

        grad1 = np.gradient(signal1)
        grad1 /= grad1.max()

        grad2 = np.gradient(signal2)
        grad2 /= grad2.max()

        phase1 = np.argmin(grad1)
        phase2 = np.argmax(grad2) + min_index
        return phase1, phase2

    def _objective_function(self, s, signal, phase1, phase2):
        delta = self.cfg['MODEL_OPTIMIZATION_DELTA']
        power = self.cfg['MODEL_POLYNOMIAL_DEGREE']

        if phase1 - delta <= 0 or phase2 + delta >= len(signal) or (phase2 - delta) - (phase1 + delta) < 5:
            delta = 2

        y = np.concatenate([
            signal[: phase1 - delta],
            signal[phase1 + delta : phase2 - delta] * (1 + s),
            signal[phase2 + delta :]
        ])
        x = np.arange(len(y))

        coeffs = np.polyfit(x, y, deg=power)
        poly = np.poly1d(coeffs)
        error = np.abs(poly(x) - y).mean()
        return error

    # ---- per-frequency feature extraction ----
    def _predict_single_channel(self, signal_1d):
        # 1) smooth
        signal_1d_smooth = savgol_filter(signal_1d, 20, 2)
        T = len(signal_1d_smooth)

        # 2) phases
        phase1, phase2 = self._phase_detector(signal_1d_smooth)

        # 3) optimize s (depth)
        bounded_phase1 = max(self.cfg['MODEL_OPTIMIZATION_DELTA'], phase1)
        bounded_phase2 = min(T - self.cfg['MODEL_OPTIMIZATION_DELTA'] - 1, phase2)

        result = minimize(
            fun=self._objective_function,
            x0=[0.0001],
            args=(signal_1d_smooth, bounded_phase1, bounded_phase2),
            method="Nelder-Mead"
        )
        s_hat = result.x[0] if result.success else 0.0
        depth = s_hat * self.cfg['SCALE']

        # 4) side features (same logic)
        delta = self.cfg['MODEL_OPTIMIZATION_DELTA']
        if bounded_phase1 - delta <= 0 or bounded_phase2 + delta >= T or (bounded_phase2 - delta) - (bounded_phase1 + delta) < 5:
            delta = 2

        i0, i1 = bounded_phase1 + delta, bounded_phase1 - delta
        e0, e1 = bounded_phase2 - delta, bounded_phase2 + delta

        oot_mask = np.zeros(T, bool); oot_mask[:i1] = True; oot_mask[e1:] = True
        it_mask  = np.zeros(T, bool); it_mask[i0:e0] = True

        baseline  = signal_1d_smooth[oot_mask].mean() if oot_mask.any() else 1.0
        rms_out   = signal_1d_smooth[oot_mask].std() + 1e-12 if oot_mask.any() else 1e-12
        rms_in    = signal_1d_smooth[it_mask].std() if it_mask.any() else 0.0
        rms_ratio = rms_in / rms_out

        d1_smooth = np.gradient(signal_1d_smooth)
        d2_smooth = np.gradient(d1_smooth)
        w = max(3, delta * 2 + 1)
        curv_ing = np.nanmean(np.abs(d2_smooth[max(0, bounded_phase1 - w):min(T, bounded_phase1 + w)]))
        curv_egr = np.nanmean(np.abs(d2_smooth[max(0, bounded_phase2 - w):min(T, bounded_phase2 + w)]))

        eqw_slice = slice(max(0, i0), min(T, e0))
        eqw = np.sum((baseline - signal_1d_smooth[eqw_slice]) / baseline) if eqw_slice.start < eqw_slice.stop else 0.0

        return {
            "depth": float(depth),
            "baseline": float(baseline),
            "rms_in": float(rms_in),
            "rms_out": float(rms_out),
            "rms_ratio": float(rms_ratio),
            "curv_ing": float(curv_ing),
            "curv_egr": float(curv_egr),
            "eqw": float(eqw),
            "T14_frames": float(phase2 - phase1),
            "phase1": float(phase1),
            "phase2": float(phase2),
        }

    def predict_per_frequency(self, single_preprocessed_signal):
        """
        Input: (T, 1+F) or (T, F) where col0 may be time/ignored.
        Output: list of dicts length F.
        """
        # decide where frequency channels start
        if single_preprocessed_signal.ndim != 2:
            raise ValueError("Expected 2D array (T, C).")

        T, C = single_preprocessed_signal.shape
        # If there's a known leading column to ignore (like time), skip it
        start_col = 1 if C > 1 else 0
        F = C - start_col
        feats_per_freq = []
        for f in range(F):
            sig = single_preprocessed_signal[:, start_col + f]
            feats_per_freq.append(self._predict_single_channel(sig))
        return feats_per_freq  # length F, each a dict

    def predict_all(self, preprocessed_signals):
        """
        Input: (N, T, C) -> returns (features_array, feature_names)
        features_array shape: (N, F, K) with fixed feature order.
        """
        feature_keys = [
            "depth", "baseline", "rms_in", "rms_out", "rms_ratio",
            "curv_ing", "curv_egr", "eqw", "T14_frames", "phase1", "phase2"
        ]

        N, T, C = preprocessed_signals.shape
        start_col = 1 if C > 1 else 0
        F = C - start_col
        K = len(feature_keys)

        out = np.zeros((N, F, K), dtype=np.float32)

        for i in tqdm(range(N), desc="Extracting Features per-frequency"):
            feats_list = self.predict_per_frequency(preprocessed_signals[i])
            # map dicts to ordered vector
            for f in range(F):
                vec = [feats_list[f][k] for k in feature_keys]
                out[i, f, :] = np.asarray(vec, dtype=np.float32)

        return out, feature_keys


transit_model = TransitModelWithFeaturesPerFreq(config)
transit_features, _ = transit_model.predict_all(train_planets)
np.save(config['OUTPUT_DIR'] + '/transit_features.npy', transit_features)

transit_features = np.load(config['OUTPUT_DIR'] + '/transit_features.npy', allow_pickle=False)
transit_features.shape

Extracting Features per-frequency:   0%|          | 0/17 [00:00<?, ?it/s]

(17, 282, 11)

In [23]:
star_info = pd.read_csv(config['DATA_PATH'] + f"/{'train' if config['IS_TRAIN'] else 'test'}_star_info.csv", index_col='planet_id').loc[train_indices]
val_star_info = star_info.loc[val_indices]
# star_info_columns = ['Rs', 'Ms', 'Ts', 'Mp', 'e', 'P', 'sma', 'i']
star_info_columns = ['Rs', 'i']

transit_features = np.concatenate((transit_features.reshape(transit_features.shape[0], -1), star_info[star_info_columns].values), axis=1)
# val_transit_features = np.concatenate((val_transit_features.reshape(val_transit_features.shape[0], -1), val_star_info[star_info_columns].values), axis=1)
transit_features.shape

(17, 3104)

In [24]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import joblib

class MLPModel(nn.Module):
    def __init__(self, input_dim=3, output_dim=283):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            # nn.ReLU(),
            # nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, output_dim)
        )
    
    def forward(self, x):
        return self.net(x)
    
def load_mu_backbone(checkpoint_path='./checkpoints_gll_mean/gll_model_4.722469100215676e-07.pth', scaler_path='./checkpoints_gll_mean/scaler.save'):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    model = MLPModel(input_dim=checkpoint['input_dim'], output_dim=checkpoint['output_dim'])
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    scaler = joblib.load(scaler_path)
    return model, scaler

# ====== Stable normalized GLL loss (no .item, backprop-safe) ======
def normalized_gll_loss_from_sigma(preds, targets, sigma_pred,
                                   naive_mean=0.01468902, naive_sigma=0.01066135,
                                   fgs_weight=57.846, fgs_sigma_true=1e-6, airs_sigma_true=1e-5,
                                   eps=1e-12):
    """
    Returns a scalar loss = -normalized_GLL (mean-weighted over batch and wavelengths),
    suitable for backprop. All tensors are torch tensors on the same device.
    """
    preds = preds.float(); targets = targets.float()
    sigma_pred = torch.clamp(sigma_pred.float(), min=eps)

    B, F = preds.shape
    two_pi = preds.new_tensor(2 * np.pi)

    def logpdf(x, mu, sigma):
        return -0.5 * ((x - mu) / sigma) ** 2 - torch.log(sigma) - 0.5 * torch.log(two_pi)

    GLL_pred = logpdf(targets, preds, sigma_pred)

    sigma_true = torch.cat([
        preds.new_full((1,), fgs_sigma_true),
        preds.new_full((F-1,), airs_sigma_true)
    ], dim=0).view(1, F).expand(B, -1)
    GLL_true = logpdf(targets, targets, sigma_true)
    GLL_mean = logpdf(targets,
                      preds.new_full((B, F), naive_mean),
                      preds.new_full((B, F), naive_sigma))

    denom = torch.clamp(GLL_true - GLL_mean, min=eps)
    ind_scores = (GLL_pred - GLL_mean) / denom  # (B,F)

    w = torch.cat([
        preds.new_full((1,), fgs_weight),
        torch.ones(F-1, device=preds.device)
    ], dim=0).view(1, F).expand_as(ind_scores)

    # Average like official metric
    score = (ind_scores * w).sum() / w.sum()  # scalar
    loss = -score  # maximize score -> minimize negative score
    return loss, score  # return both for logging


# ====== Frozen trunk + sigma head with instrument floor ======
class TrunkSigmaHead(nn.Module):
    def __init__(self, mu_model: nn.Module, out_dim=283, head_hidden=256,
                 fgs_floor=1e-6, airs_floor=1e-5, trunk_freeze=True):
        super().__init__()
        layers = list(mu_model.net.children())
        assert isinstance(layers[-1], nn.Linear), "Last layer must be Linear in the μ model"
        self.trunk = nn.Sequential(*layers[:-1])
        for p in self.trunk.parameters():
            p.requires_grad = not trunk_freeze

        trunk_out = layers[-1].in_features
        self.head = nn.Sequential(
            nn.Linear(trunk_out, head_hidden),
            # nn.GELU(),
            # nn.Linear(head_hidden, head_hidden),
            # nn.GELU(),
            # nn.Dropout(0.1),
            # nn.Linear(head_hidden, head_hidden),
            # nn.GELU(),
            # nn.Dropout(0.15),
            # nn.Linear(head_hidden, head_hidden),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(head_hidden, out_dim)
        )
        floor = torch.cat([
            torch.tensor([fgs_floor], dtype=torch.float32),
            torch.full((out_dim-1,), airs_floor, dtype=torch.float32)
        ], dim=0)
        self.register_buffer("sigma_floor", floor)
        self.softplus = nn.Softplus()

    def forward(self, x):
        raw = self.head(self.trunk(x))
        sp  = self.softplus(raw)
        sigma = torch.sqrt(sp**2 + self.sigma_floor.view(1, -1)**2)
        return sigma


# ===== Dataset: returns (X, Y, mu_hat) =====
class SigmaGLLDataset(Dataset):
    def __init__(self, X_scaled: np.ndarray, Y: np.ndarray, mu_hat: np.ndarray):
        assert X_scaled.shape[0] == Y.shape[0] == mu_hat.shape[0], "Row count mismatch"
        self.X = torch.from_numpy(X_scaled.astype(np.float32))
        self.Y = torch.from_numpy(Y.astype(np.float32))
        self.M = torch.from_numpy(mu_hat.astype(np.float32))
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.Y[i], self.M[i]

# ===== Helper: align mu_predictions to train rows =====
def load_mu_predictions(mu_predictions_path, row_ids):
    """
    mu_predictions_path: CSV with index planet_id and 283 columns (lambda_*)
    row_ids: array-like of planet_id for each row in train_X/train_Y order
    returns mu_hat aligned to row_ids (N, 283)
    """
    df = pd.read_csv(mu_predictions_path, index_col='planet_id')
    # If columns are not exactly 283 named, sort or select first 283 numeric columns
    mu_hat = df.reindex(row_ids).values.astype(np.float32)
    if mu_hat.shape[1] != 283:
        raise ValueError(f"Expected 283 mu columns, got {mu_hat.shape[1]}")
    return mu_hat


# ===== Trainer that uses offline mu_hat for loss/metrics =====
def train_sigma_head(
        train_X, train_Y, train_planet_ids,
        mu_checkpoint_path='./checkpoints_gll_mean/gll_model_4.722469100215676e-07.pth',
        scaler_path='./checkpoints_gll_mean/scaler.save',
        mu_predictions_path='./predictions/mu_predictions.csv',
        num_epochs=200, batch_size=512, lr=1e-3, weight_decay=1e-5,
        fgs_weight=57.846,
        naive_mean=0.01468902, naive_sigma=0.01066135,
        fgs_sigma_true=1e-6, airs_sigma_true=1e-5,
        val_ratio=0.25,
        ckpt_dir="./checkpoints_gll_mean",
        fgs_floor=1e-6, airs_floor=1e-5,
        lr_patience=100,
        trunk_freeze=True
    ):
    os.makedirs(ckpt_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1) Load μ backbone (for trunk init only) and scaler
    mu_backbone, scaler = load_mu_backbone(mu_checkpoint_path, scaler_path)

    # 2) Scale features
    train_X_scaled = scaler.transform(train_X).astype(np.float32)

    # 3) Load offline μ predictions aligned to rows
    mu_hat_all = load_mu_predictions(mu_predictions_path, train_planet_ids)
    # 4) Split indices 3:1 (train:val)
    N = train_X_scaled.shape[0]
    idx = np.arange(N)
    rng = np.random.default_rng(42)
    rng.shuffle(idx)
    val_n = int(N * val_ratio)
    val_idx = idx[:val_n]
    trn_idx = idx[val_n:]

    # 5) Build datasets/dataloaders
    trn_ds = SigmaGLLDataset(train_X_scaled[trn_idx], train_Y[trn_idx], mu_hat_all[trn_idx])
    val_ds = SigmaGLLDataset(train_X_scaled[val_idx], train_Y[val_idx], mu_hat_all[val_idx])
    trn_dl = DataLoader(trn_ds, batch_size=batch_size, shuffle=True,  num_workers=0)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    # 6) Sigma model
    sigma_model = TrunkSigmaHead(mu_backbone, out_dim=283,
                                       head_hidden=256, fgs_floor=fgs_floor, airs_floor=airs_floor, trunk_freeze=trunk_freeze).to(device)
    opt = optim.AdamW(filter(lambda p: p.requires_grad, sigma_model.parameters()),
                      lr=lr, weight_decay=weight_decay)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=lr_patience, threshold=1e-7)

    best_val = float("inf"); best_path = os.path.join(ckpt_dir, "sigma_head_best.pt")

    for ep in range(1, num_epochs+1):
        # ---- Train
        sigma_model.train()
        tot_loss = tot_score = n = 0.0
        for xb, yb, mb in trn_dl:
            xb = xb.to(device); yb = yb.to(device); mb = mb.to(device)
            opt.zero_grad()
            sigma_b = sigma_model(xb)
            loss_b, score_b = normalized_gll_loss_from_sigma(
                mb, yb, sigma_b,
                naive_mean=naive_mean, naive_sigma=naive_sigma,
                fgs_weight=fgs_weight,
                fgs_sigma_true=fgs_sigma_true, airs_sigma_true=airs_sigma_true
            )
            loss_b.backward(); opt.step()
            bs = xb.size(0)
            tot_loss += loss_b.item() * bs
            tot_score += score_b.item() * bs
            n += bs
        train_loss = tot_loss / max(1, n)
        train_gll  = tot_score / max(1, n)

        # ---- Validate
        sigma_model.eval()
        with torch.no_grad():
            v_tot_loss = v_tot_score = vn = 0.0
            for xv, yv, mv in val_dl:
                xv = xv.to(device); yv = yv.to(device); mv = mv.to(device)
                sigma_v = sigma_model(xv)
                v_loss, v_score = normalized_gll_loss_from_sigma(
                    mv, yv, sigma_v,
                    naive_mean=naive_mean, naive_sigma=naive_sigma,
                    fgs_weight=fgs_weight,
                    fgs_sigma_true=fgs_sigma_true, airs_sigma_true=airs_sigma_true
                )
                bs = xv.size(0)
                v_tot_loss += v_loss.item() * bs
                v_tot_score += v_score.item() * bs
                vn += bs
            val_loss = v_tot_loss / max(1, vn)
            val_gll  = v_tot_score / max(1, vn)

        sched.step(val_loss)

        # Save best
        if val_loss < best_val:
            best_val = val_loss
            torch.save({"sigma_model": sigma_model.state_dict()}, best_path)

        if (ep % 10 == 0) or (ep == 1) or (ep == num_epochs):
            print(f"Epoch {ep:4d} | train_loss=-GLL {train_loss:.6e} | train_GLL {train_gll:.4f} | val_loss=-GLL {val_loss:.6e} | val_GLL {val_gll:.4f}")

    # Load best
    if os.path.exists(best_path):
        sigma_state = torch.load(best_path, map_location="cpu")["sigma_model"]
        sigma_model.load_state_dict(sigma_state)

    return sigma_model, scaler


@torch.no_grad()
def predict_sigma_and_save(sigma_model, scaler, X_all, planet_ids,
                           mu_predictions_path, out_csv_path='./predictions/submission.csv'):
    device = next(sigma_model.parameters()).device
    X_scaled = scaler.transform(X_all).astype(np.float32)
    mu_predictions = load_mu_predictions(mu_predictions_path, planet_ids)
    print(mu_predictions)
    sigma_model = sigma_model.to(device).eval()

    dataloader = DataLoader(torch.from_numpy(X_scaled), batch_size=64, shuffle=False, num_workers=0)
    sigma_predictions = []
    for batch_X in dataloader:
        batch_X = batch_X.to(device).float()
        batch_sigmas = sigma_model(batch_X).cpu().numpy()
        sigma_predictions.append(batch_sigmas)
    sigma_predictions = np.concatenate(sigma_predictions, axis=0)

    sample_sub = pd.read_csv(config['DATA_PATH'] + '/sample_submission.csv', index_col='planet_id')
    submission = pd.DataFrame(index=pd.Series(planet_ids, name='planet_id'), columns=sample_sub.columns, dtype=np.float32)
    submission.iloc[:, :283]  = mu_predictions
    submission.iloc[:, 283:]  = np.clip(sigma_predictions, 1e-15, None)
    submission.sort_index(inplace=True)

    submission.to_csv(out_csv_path)
    print(f"Wrote {out_csv_path}")

In [25]:
# sigma_model = train_sigma_head(
#     checkpoint_path='./checkpoints_gll_mean/gll_model_4.722469100215676e-07.pth',
#     scaler_path='./checkpoints_gll_mean/scaler.pkl',
#     mu_predictions_path='./predictions/mu_predictions.csv',
#     train_X=train_transit_features.astype(np.float32),
#     train_Y=train_labels.astype(np.float32),
#     train_planet_ids=train_indices,
#     val_ratio=0.2,
#     num_epochs=1000,
#     batch_size=128,
#     lr=1e-2,
#     ckpt_dir="./checkpoints_gll_mean",
#     lr_patience=200,
#     trunk_freeze=True
# )

In [27]:
mu_backbone_checkpoint_path = './checkpoints_gll_mean/gll_model_4.722469100215676e-07.pth'
sigma_checkpoint_path = "./checkpoints_gll_mean/sigma_head_best.pt"
scaler_path = './checkpoints_gll_mean/scaler.pkl'
mu_backbone, scaler = load_mu_backbone(mu_backbone_checkpoint_path, scaler_path)
mu_predictions_path = './predictions/mu_predictions.csv'

sigma_checkpoint = torch.load(sigma_checkpoint_path, map_location='cpu')
model = TrunkSigmaHead(mu_model=mu_backbone, out_dim=283)
model.load_state_dict(sigma_checkpoint['sigma_model'])


predict_sigma_and_save(
    sigma_model=model,
    scaler=scaler,
    X_all=transit_features.astype(np.float32),
    planet_ids=train_indices,
    mu_predictions_path=mu_predictions_path,
    out_csv_path='./predictions/submission.csv'
)


[[0.00551587 0.00551715 0.00551788 ... 0.00551252 0.00551704 0.00551663]
 [0.00867006 0.00867131 0.00867203 ... 0.00866676 0.00867121 0.00867071]
 [0.01069684 0.01069806 0.01069856 ... 0.01069363 0.01069796 0.01069763]
 ...
 [0.0054085  0.0054098  0.00541048 ... 0.0054052  0.0054097  0.00540927]
 [0.00511273 0.00511404 0.00511467 ... 0.00510938 0.00511389 0.00511354]
 [0.01486554 0.01486669 0.01486725 ... 0.01486253 0.01486662 0.01486634]]
Wrote ./predictions/submission.csv


C:\Users\fazul\AppData\Local\Temp\ipykernel_4848\384207349.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location='cpu')


In [28]:
pd.read_csv('./predictions/submission.csv', index_col='planet_id')

,wl_1,wl_2,wl_3,wl_4,wl_5,wl_6,wl_7,wl_8,wl_9,wl_10,...,sigma_274,sigma_275,sigma_276,sigma_277,sigma_278,sigma_279,sigma_280,sigma_281,sigma_282,sigma_283
planet_id,,,,,,,,,,,,,,,,,,,,,
104891231,0.021442,0.021443,0.021443,0.021441,0.021441,0.021443,0.021441,0.021445,0.021442,0.021442,...,0.000458,0.000458,0.000461,0.000458,0.000464,0.000459,0.000466,0.000468,0.000474,0.000486
1010375142,0.005516,0.005517,0.005518,0.005515,0.005516,0.005517,0.005516,0.005520,0.005517,0.005516,...,0.000340,0.000333,0.000331,0.000339,0.000330,0.000346,0.000331,0.000319,0.000316,0.000315
1024292144,0.008670,0.008671,0.008672,0.008670,0.008670,0.008671,0.008670,0.008674,0.008671,0.008670,...,0.000331,0.000326,0.000327,0.000329,0.000326,0.000333,0.000325,0.000319,0.000319,0.000321
1029552010,0.010697,0.010698,0.010699,0.010696,0.010696,0.010698,0.010697,0.010701,0.010697,0.010697,...,0.000260,0.000258,0.000259,0.000259,0.000258,0.000261,0.000258,0.000256,0.000258,0.000262
1031303815,0.020266,0.020267,0.020268,0.020266,0.020266,0.020267,0.020266,0.020270,0.020266,0.020266,...,0.000769,0.000758,0.000763,0.000821,0.000764,0.000846,0.000778,0.000760,0.000765,0.000778
1042982756,0.015451,0.015452,0.015453,0.015451,0.015451,0.015452,0.015451,0.015455,0.015452,0.015451,...,0.000514,0.000511,0.000518,0.000537,0.000521,0.000552,0.000526,0.000510,0.000522,0.000536
1047977648,0.008935,0.008936,0.008937,0.008935,0.008935,0.008936,0.008935,0.008939,0.008935,0.008935,...,0.000211,0.000209,0.000209,0.000210,0.000208,0.000212,0.000207,0.000204,0.000204,0.000206
1048114509,0.012146,0.012147,0.012148,0.012146,0.012146,0.012147,0.012146,0.012150,0.012147,0.012146,...,0.000215,0.000213,0.000215,0.000214,0.000215,0.000215,0.000213,0.000213,0.000216,0.000218
1049092982,0.017343,0.017344,0.017345,0.017343,0.017343,0.017345,0.017343,0.017347,0.017344,0.017344,...,0.000387,0.000384,0.000388,0.000397,0.000392,0.000406,0.000401,0.000394,0.000393,0.000405
